# Two-Stage Voice Cloning: Base TTS + RVC (Python 3.10 Environment)
### Audio Deepfake Detection Project — RVC Training & Inference Module

> **Why we use a custom Python 3.10 environment:**
> Colab runs Python 3.13 by default. RVC's core dependencies (`fairseq`, `pyworld`,
> `praat-parselmouth`) require compiled C-extensions with no Python 3.13 wheels.
> We install Miniconda and create an isolated Python 3.10 environment where all
> dependencies install cleanly without compilation failures.
>
> **Attribution:**
> - Pretrained RVC Architecture & HuBERT base: RVC-Project (not our work)
> - Custom RVC voice model weights: trained on our consenting team member's audio (our work)
> - Integration & detection pipeline: our work

## Cell 1: Verify GPU and Install Miniconda with Python 3.10
**Run this cell first. It takes approximately 2–3 minutes.**

In [ ]:
import os, sys, subprocess

# Step A: Confirm GPU is active
!nvidia-smi

# Step B: Download and install Miniconda silently
if not os.path.exists('/opt/conda/bin/conda'):
    print("Installing Miniconda...")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -u -p /opt/conda
    !rm /tmp/miniconda.sh
    print("[OK] Miniconda installed.")
else:
    print("[OK] Miniconda already installed.")

# Step C: Add conda to PATH for this session
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

# Step D: Create isolated Python 3.10 environment for RVC
env_name = 'rvc_env'
env_path = f'/opt/conda/envs/{env_name}'

if not os.path.exists(env_path):
    print(f"Creating Python 3.10 conda environment '{env_name}'...")
    !conda create -n {env_name} python=3.10 -y -q
    print(f"[OK] Environment '{env_name}' created.")
else:
    print(f"[OK] Environment '{env_name}' already exists.")

# Shorthand paths for use in all subsequent cells
PY  = f'{env_path}/bin/python'
PIP = f'{env_path}/bin/pip'

print(f"\nPython path : {PY}")
print(f"Pip path    : {PIP}")
!{PY} --version

## Cell 2: Install PyTorch (CUDA) + Core RVC Dependencies in Python 3.10 Environment
**Expected duration: 4–6 minutes. CUDA-enabled torch is required for GPU training.**

In [ ]:
import os
PY  = '/opt/conda/envs/rvc_env/bin/python'
PIP = '/opt/conda/envs/rvc_env/bin/pip'

# 1. Install PyTorch 2.1 with CUDA 12.1 (compatible with Colab T4 driver)
print("Installing PyTorch 2.1 + CUDA 12.1...")
!{PIP} install -q torch==2.1.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121

# 2. Install fairseq (builds cleanly on Python 3.10)
print("Installing fairseq...")
!{PIP} install -q fairseq

# 3. Install pyworld (pitch extraction, WORLD vocoder interface)
print("Installing pyworld...")
!{PIP} install -q pyworld

# 4. Install remaining RVC audio and ML dependencies
print("Installing remaining RVC dependencies...")
!{PIP} install -q \
    praat-parselmouth \
    torchcrepe \
    faiss-gpu \
    librosa \
    soundfile \
    scipy \
    pydub \
    ffmpeg-python \
    edge-tts

# 5. Verify all imports work
print("\nVerifying critical imports...")
!{PY} -c "
import torch
import torchaudio
import pyworld
import parselmouth
import fairseq
import faiss
import librosa
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print('[SUCCESS] All RVC dependencies imported cleanly in Python 3.10!')
"

## Cell 3: Clone RVC Repository & Download Pretrained Base Weights
**Downloads ~1.5 GB total (HuBERT + RVC v2 base generator/discriminator + RMVPE pitch estimator)**

In [ ]:
import os
PIP = '/opt/conda/envs/rvc_env/bin/pip'

# Clone RVC-WebUI repository
if not os.path.exists('/content/rvc'):
    !git clone -q https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/rvc
    print("[OK] RVC repository cloned.")
else:
    print("[OK] RVC repository already cloned.")

# Install RVC's own requirements inside the py310 environment
!{PIP} install -q -r /content/rvc/requirements.txt

# Create required asset directories
!mkdir -p /content/rvc/assets/hubert
!mkdir -p /content/rvc/assets/pretrained_v2
!mkdir -p /content/rvc/assets/rmvpe

# Download HuBERT base (Content Encoder - extracts what words are spoken)
if not os.path.exists('/content/rvc/assets/hubert/hubert_base.pt'):
    print("Downloading HuBERT base model (~189 MB)...")
    !wget -q -O /content/rvc/assets/hubert/hubert_base.pt \
        https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt

# Download RVC v2 40kHz pretrained generator (voice synthesizer backbone)
if not os.path.exists('/content/rvc/assets/pretrained_v2/f0G40k.pth'):
    print("Downloading RVC v2 pretrained Generator (~192 MB)...")
    !wget -q -O /content/rvc/assets/pretrained_v2/f0G40k.pth \
        https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth

# Download RVC v2 40kHz pretrained discriminator (for adversarial training)
if not os.path.exists('/content/rvc/assets/pretrained_v2/f0D40k.pth'):
    print("Downloading RVC v2 pretrained Discriminator (~196 MB)...")
    !wget -q -O /content/rvc/assets/pretrained_v2/f0D40k.pth \
        https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth

# Download RMVPE (Robust Model for Vocal Pitch Estimation - replaces old crepe pitch tracker)
if not os.path.exists('/content/rvc/assets/rmvpe/rmvpe.pt'):
    print("Downloading RMVPE pitch estimator (~181 MB)...")
    !wget -q -O /content/rvc/assets/rmvpe/rmvpe.pt \
        https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt

# Final check
files_needed = [
    '/content/rvc/assets/hubert/hubert_base.pt',
    '/content/rvc/assets/pretrained_v2/f0G40k.pth',
    '/content/rvc/assets/pretrained_v2/f0D40k.pth',
    '/content/rvc/assets/rmvpe/rmvpe.pt'
]
print("\n=== Asset Verification ===")
all_ok = True
for f in files_needed:
    size_mb = os.path.getsize(f) / (1024*1024) if os.path.exists(f) else 0
    status = f'OK  ({size_mb:.1f} MB)' if os.path.exists(f) and size_mb > 10 else 'MISSING or CORRUPT'
    print(f"  {os.path.basename(f):30s} : {status}")
    if 'MISSING' in status:
        all_ok = False

print()
if all_ok:
    print("[SUCCESS] All base models downloaded. Ready for Step 3 (Training Data Upload).")
else:
    print("[WARNING] Some files missing. Re-run this cell.")